# 06 — Route choice: choice sets and path-size logit

Equilibrium assignment assumes drivers only ever take cost-minimal routes. **Route
choice models** instead generate a *set* of plausible routes per OD pair and split
demand among them with a discrete-choice model. AequilibraE implements:

- **BFSLE** (breadth-first search with link elimination) and **link penalisation**
  for choice-set generation — both extremely fast Cython implementations;
- **Path-Size Logit (PSL)** assignment, which corrects the IIA problem for
  overlapping routes.

We use the Coquimbo model, with utility = distance × θ.


In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import numpy as np

from aequilibrae.utils.create_example import create_example

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "coquimbo")

theta = 0.00011                       # utility per metre
nodes_of_interest = (71645, 74089, 77011, 79385)

project.network.build_graphs()
graph = project.network.graphs["c"]
graph.network = graph.network.assign(utility=graph.network.distance * theta)
graph.prepare_graph(np.array(nodes_of_interest))
graph.set_graph("utility")

C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: UserWarning: Found centroids not present in the graph!
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133]
  build_compressed_graph(self, remove_dead_ends)
C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the i

C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  build_compressed_graph(self, remove_dead_ends)


In [2]:
# A small synthetic demand matrix between our nodes of interest
from aequilibrae.matrix import AequilibraeMatrix

mat = AequilibraeMatrix()
mat.create_empty(zones=graph.num_zones, matrix_names=["demand"], memory_only=True)
mat.index = graph.centroids[:]
mat.matrices[:, :, 0] = np.full((graph.num_zones, graph.num_zones), 10.0)
mat.computational_view()

In [3]:
from aequilibrae.paths import RouteChoice

rc = RouteChoice(graph)
rc.add_demand(mat)

# Always bound the generation: at most 5 routes per OD pair here
rc.set_choice_set_generation("bfsle", max_routes=5)
rc.default_parameters

{'generic': {'seed': 0,
  'max_routes': 0,
  'max_depth': 0,
  'max_misses': 100,
  'penalty': 1.01,
  'cutoff_prob': 0.0,
  'beta': 1.0,
  'store_results': True},
 'link-penalisation': {},
 'bfsle': {'penalty': 1.0}}

In [4]:
# Generate a choice set for one OD pair and assign its demand with PSL
results = rc.execute_single(77011, 74089, demand=1.0)
print(f"{len(results)} routes found between 77011 and 74089")
results[0][:12]  # link ids of the first route

5 routes found between 77011 and 74089


(np.int64(-24222),
 np.int64(30332),
 np.int64(30333),
 np.int64(-10435),
 np.int64(30068),
 np.int64(30069),
 np.int64(14198),
 np.int64(14199),
 np.int64(31161),
 np.int64(30928),
 np.int64(-31622),
 np.int64(24112))

In [5]:
# JupyterGIS map helper ------------------------------------------------------
# GISDocument is JupyterGIS' notebook API: it builds a live, QGIS-like map
# document rendered directly in JupyterLab. Layers added from GeoDataFrames
# are converted to GeoJSON on the fly.
#
# add_gdf also translates the declarative symbology into the OpenLayers
# flat-style expressions the current JupyterGIS frontend renders from, so
# colours and line widths show up without touching the symbology panel.
import json

import matplotlib.colors
import matplotlib.pyplot as _plt
from jupytergis import GISDocument
from jupytergis_lab.notebook.symbology import to_symbology_state

OSM_TILES = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"

def new_map(gdf_for_extent=None, zoom=12):
    """Create a GISDocument centred on a layer, with an OpenStreetMap basemap."""
    kwargs = {}
    if gdf_for_extent is not None:
        b = gdf_for_extent.total_bounds  # (minx, miny, maxx, maxy)
        kwargs = {"longitude": (b[0] + b[2]) / 2, "latitude": (b[1] + b[3]) / 2, "zoom": zoom}
    doc = GISDocument(**kwargs)
    doc.add_raster_layer(OSM_TILES, name="OpenStreetMap", attribution="(C) OpenStreetMap contributors", opacity=0.6)
    return doc

def _hex(rgba):
    return matplotlib.colors.to_hex(rgba)

def _ramp_expr(fld, params):
    name, dom = params.get("name", "viridis"), params.get("domain") or [0.0, 1.0]
    cmap = _plt.get_cmap(name)
    if params.get("reverse"):
        cmap = cmap.reversed()
    expr = ["interpolate", ["linear"], ["get", fld]]
    for i in range(7):
        t = i / 6
        expr += [dom[0] + t * (dom[1] - dom[0]), _hex(cmap(t))]
    return expr

def _scalar_expr(fld, params):
    d, r = params["domain"], params["range"]
    return ["interpolate", ["linear"], ["get", fld], d[0], r[0], d[1], r[1]]

def _cat_expr(fld, params, gdf):
    cmap = _plt.get_cmap(params.get("colorRamp", "tab10"))
    vals = list(dict.fromkeys(gdf[fld].dropna()))
    expr = ["match", ["get", fld]]
    for i, v in enumerate(vals):
        expr += [v, _hex(cmap(i % cmap.N))]
    return expr + ["#9ca3af"]

def _flat_style(symbology, gdf):
    """Grammar symbology -> OpenLayers flat-style dict (what the map renders)."""
    state = to_symbology_state(symbology)
    if not state:
        return None
    flat = {}
    for layer in state.get("layers", []):
        for rule in layer.get("rules", []):
            flds = rule.get("fields") or [None]
            for m in rule.get("mappings", []):
                scheme = m["scale"]["scheme"]
                params = m["scale"].get("params", {})
                if scheme == "constant_rgba":
                    val = params["value"]
                    val = _hex([c if c <= 1 else c / 255 for c in val]) if isinstance(val, (list, tuple)) else val
                elif scheme == "constant_num":
                    val = params["value"]
                elif scheme == "colorMap":
                    val = _ramp_expr(flds[0], params)
                elif scheme == "scalar":
                    val = _scalar_expr(flds[0], params)
                elif scheme == "categorical":
                    val = _cat_expr(flds[0], params, gdf)
                else:
                    continue
                for enc in m.get("encodings", []):
                    flat[enc] = val
    if any(k.startswith("circle") for k in flat) and "circle-radius" not in flat:
        flat["circle-radius"] = 5
    if "stroke-color" in flat and "stroke-width" not in flat:
        flat["stroke-width"] = 1.5
    return flat or None

def merge_lines(gdf, tol=0.01):
    """Collapse many lines into a single MultiLineString feature.

    Backdrop and class-display layers do not need per-feature identity, and one
    merged feature is a fraction of the size of tens of thousands of features -
    which keeps national-scale layers inside the notebook sync message limit.
    """
    import geopandas as _gpd
    from shapely.geometry import MultiLineString
    parts = []
    for geom in gdf.geometry.simplify(tol):
        if geom is None or geom.is_empty:
            continue
        parts.extend(geom.geoms if geom.geom_type == "MultiLineString" else [geom])
    return _gpd.GeoDataFrame({"links": [len(parts)]}, geometry=[MultiLineString(parts)], crs=gdf.crs)

def _round_coords(o, nd=5):
    if isinstance(o, (int, float)):
        return round(o, nd)
    if isinstance(o, list):
        return [_round_coords(v, nd) for v in o]
    return o

def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a GeoJSON layer, with rendered symbology.

    Coordinates are quantized to ~1 m so even national-scale layers stay well
    under Jupyter's websocket message limit (oversized layers are dropped
    silently by the sync, so this matters more than it looks).
    """
    data = json.loads(gdf.to_json())
    for f in data.get("features", []):
        g = f.get("geometry")
        if g and "coordinates" in g:
            g["coordinates"] = _round_coords(g["coordinates"])
    lid = doc.add_geojson_layer(data=data, name=name,
                                symbology=symbology, **kwargs)
    flat = _flat_style(symbology, gdf)
    if flat:
        layer = doc._layers.get(lid)
        layer["parameters"]["color"] = flat
        doc._layers[lid] = layer
    return lid

In [6]:
from jupytergis_lab.notebook.symbology import constant

links = project.network.links.data
palette = ["#dc2626", "#2563eb", "#16a34a", "#d97706", "#7c3aed"]

route_links = links[links.link_id.isin({l for route in results for l in route})]
doc = new_map(route_links, zoom=13)
add_gdf(doc, links, "network", opacity=0.4, symbology=[[constant("#94a3b8").encoding("stroke")]])
for i, route in enumerate(results):
    add_gdf(doc, links[links.link_id.isin(route)], f"route {i + 1}",
            symbology=[[constant(palette[i % len(palette)]).encoding("stroke")]])
doc

C:\Users\Riz\AppData\Local\Temp\ipykernel_18712\1581409630.py:24: UserWarning: The JupyterGIS Python API is better experienced in the xeus-python kernel which supports awaiting comm messages
  doc = GISDocument(**kwargs)


## Batch assignment

`prepare()` + `execute(perform_assignment=True)` runs generation and PSL assignment
for **every** OD pair in the demand matrix, giving link-level loads.


In [7]:
rc.prepare()
rc.execute(perform_assignment=True)

loads = rc.get_load_results()
loads.sort_values("demand_tot", ascending=False).head()

,demand_ab,demand_ba,demand_tot
link_id,,,
20964,27.121247,33.905817,61.027064
20963,27.121247,33.905817,61.027064
20965,27.121247,33.905817,61.027064
20962,27.121247,33.905817,61.027064
29899,27.121247,33.905817,61.027064


In [8]:
from jupytergis_lab.notebook.symbology import field

loaded = links.merge(loads.reset_index(), on="link_id")
loaded = loaded[loaded["demand_tot"] > 0]

doc2 = new_map(loaded, zoom=12)
vmax = float(loaded["demand_tot"].max())
add_gdf(doc2, loaded[["link_id", "demand_tot", "geometry"]], "route-choice flows",
        symbology=[[field("demand_tot").colormap("viridis", domain=(0.0, vmax)).encoding("stroke"),
                field("demand_tot").scalar(domain=(0.0, vmax), output_range=(0.8, 6.5)).encoding("stroke-width")]])
doc2

In [9]:
project.close()

INFO:aequilibrae:Closed project on C:\Users\Riz\AppData\Local\Temp\1529f4aa9b3a4a8ca7b3cb74763fd60e


---
**Next:** [07 — Public transport](07_public_transport.ipynb): importing GTFS and building
a transit model.
